# PostgreSQL Load and Database Setup

This notebook loads the processed health access reporting tables into PostgreSQL.

The processed CSV files were created in `02_data_cleaning.ipynb`.

The purpose of this step is to create a structured database layer for SQL analysis and Power BI reporting.

## Load Required Libraries

Python libraries are imported for reading CSV files and connecting to PostgreSQL.

In [6]:
import sys
print(sys.executable)
print(sys.version)

/opt/anaconda3/envs/myenv/bin/python
3.8.19 (default, Mar 20 2024, 15:27:52) 
[Clang 14.0.6 ]


In [1]:
%pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

## Read Processed Data Files

The processed CSV files created in the data cleaning step are loaded into pandas DataFrames before being imported into PostgreSQL.

In [3]:
processed_path = Path("../data/processed")

dim_indicator = pd.read_csv(processed_path / "dim_indicator.csv")
dim_population_group = pd.read_csv(processed_path / "dim_population_group.csv")
fact_prevalence = pd.read_csv(processed_path / "fact_prevalence.csv")
fact_subgroup_comparison = pd.read_csv(processed_path / "fact_subgroup_comparison.csv")
fact_trend = pd.read_csv(processed_path / "fact_trend.csv")

print("Processed CSV files loaded successfully.")

Processed CSV files loaded successfully.


## Check Processed Table Shapes

The shape of each processed table is checked before loading into PostgreSQL.

This confirms that the expected files were loaded correctly and that no table is empty.

In [4]:
tables = {
    "dim_indicator": dim_indicator,
    "dim_population_group": dim_population_group,
    "fact_prevalence": fact_prevalence,
    "fact_subgroup_comparison": fact_subgroup_comparison,
    "fact_trend": fact_trend
}

for table_name, df in tables.items():
    print(table_name, df.shape)

dim_indicator (7, 6)
dim_population_group (108, 4)
fact_prevalence (2600, 19)
fact_subgroup_comparison (186, 12)
fact_trend (7448, 11)


## Connect to PostgreSQL

A database connection is created using SQLAlchemy.

This connection allows Python to load the processed reporting tables into PostgreSQL.

In [ ]:
username = "postgres"
password = "password"
host = "localhost"
port = "5432"
database = "health_access_db"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

print("Database connection created.")

Database connection created.


## Test Database Connection

A simple SQL query is used to confirm that Python can connect to the PostgreSQL database.

In [6]:
test_query = "SELECT version();"

pd.read_sql(test_query, engine)

,version
0,"PostgreSQL 18.4 on x86_64-apple-darwin24.6.0, ..."


## Load Processed Tables into PostgreSQL

The five processed reporting tables are loaded into PostgreSQL.

These tables form the database layer for SQL analysis and Power BI reporting.

In [7]:
for table_name, df in tables.items():
    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )

print("All processed tables loaded into PostgreSQL successfully.")

All processed tables loaded into PostgreSQL successfully.


## Validate Loaded PostgreSQL Tables

Row counts are checked after loading the tables into PostgreSQL.

This confirms that the database tables match the processed CSV files.

In [8]:
for table_name in tables.keys():
    query = f"SELECT COUNT(*) AS row_count FROM {table_name};"
    result = pd.read_sql(query, engine)
    print(table_name, result.loc[0, "row_count"])

dim_indicator 7
dim_population_group 108
fact_prevalence 2600
fact_subgroup_comparison 186
fact_trend 7448


## Check PostgreSQL Table List

The PostgreSQL information schema is queried to confirm that all reporting tables exist in the database.

In [9]:
table_list_query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

pd.read_sql(table_list_query, engine)

,table_name
0,dim_indicator
1,dim_population_group
2,fact_prevalence
3,fact_subgroup_comparison
4,fact_trend


## Create SQL Views

SQL views are created in PostgreSQL to support common reporting and dashboard analysis.

Views do not change the underlying tables. They save reusable SQL logic for Power BI and later analysis.

In [10]:
from sqlalchemy import text

### View 1: Latest Prevalence

This view returns the latest available total-group prevalence records for each selected health access barrier.

It supports the dashboard overview page by showing the most recent prevalence value and estimated affected population.

In [12]:
create_latest_prevalence_view = """
DROP VIEW IF EXISTS vw_latest_prevalence;

CREATE VIEW vw_latest_prevalence AS
SELECT
    di.indicator_id,
    fp.indicator_name,
    di.indicator_short_name,
    di.barrier_type,
    fp.population,
    fp.year,
    fp."group",
    fp.total_value,
    fp.total_low_ci,
    fp.total_high_ci,
    fp.estimated_number,
    fp.estimated_number_low_ci,
    fp.estimated_number_high_ci
FROM fact_prevalence fp
LEFT JOIN dim_indicator di
    ON fp.indicator_name = di.indicator_name
WHERE fp."group" = 'Total'
  AND fp.year = (
      SELECT MAX(year)
      FROM fact_prevalence
  );
"""

with engine.begin() as conn:
    conn.execute(text(create_latest_prevalence_view))

print("vw_latest_prevalence created successfully.")

vw_latest_prevalence created successfully.


In [13]:
latest_prevalence = pd.read_sql(
    """
    SELECT *
    FROM vw_latest_prevalence
    ORDER BY barrier_type, indicator_short_name;
    """,
    engine
)

latest_prevalence

,indicator_id,indicator_name,indicator_short_name,barrier_type,population,year,group,total_value,total_low_ci,total_high_ci,estimated_number,estimated_number_low_ci,estimated_number_high_ci
0,6,Unmet need for dental health care due to cost,Dental cost barrier,Cost,adults,2024,Total,43.0,41.4,44.5,1775000.0,1713000.0,1836000.0
1,6,Unmet need for dental health care due to cost,Dental cost barrier,Cost,children,2024,Total,3.0,2.1,4.2,27000.0,18000.0,37000.0
2,2,Unmet need for GP due to cost,GP cost barrier,Cost,adults,2024,Total,14.9,13.9,15.9,646000.0,601000.0,690000.0
3,2,Unmet need for GP due to cost,GP cost barrier,Cost,children,2024,Total,1.4,0.9,2.0,13000.0,8000.0,18000.0
4,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,adults,2024,Total,3.6,2.9,4.3,155000.0,125000.0,184000.0
5,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,children,2024,Total,1.0,0.6,1.5,9000.0,5000.0,13000.0
6,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,adults,2024,Total,10.5,9.6,11.5,456000.0,415000.0,497000.0
7,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,children,2024,Total,6.3,4.9,7.9,54000.0,41000.0,66000.0
8,5,Unmet need for GP due to work,GP work constraint,Time constraint,adults,2024,Total,8.8,8.0,9.7,382000.0,345000.0,420000.0
9,5,Unmet need for GP due to work,GP work constraint,Time constraint,children,2024,Total,3.4,2.5,4.4,33000.0,24000.0,42000.0


### View 2: Priority Barriers

This view ranks the latest total-group health access barriers by prevalence and estimated affected population.

It supports priority setting by identifying which access barriers affect the largest proportion and number of people.

In [14]:
create_priority_barriers_view = """
DROP VIEW IF EXISTS vw_priority_barriers;

CREATE VIEW vw_priority_barriers AS
SELECT
    di.indicator_id,
    fp.indicator_name,
    di.indicator_short_name,
    di.barrier_type,
    fp.population,
    fp.year,
    fp."group",
    fp.total_value,
    fp.estimated_number,
    RANK() OVER (
        PARTITION BY fp.population
        ORDER BY fp.total_value DESC
    ) AS prevalence_rank,
    RANK() OVER (
        PARTITION BY fp.population
        ORDER BY fp.estimated_number DESC
    ) AS estimated_number_rank
FROM fact_prevalence fp
LEFT JOIN dim_indicator di
    ON fp.indicator_name = di.indicator_name
WHERE fp."group" = 'Total'
  AND fp.year = (
      SELECT MAX(year)
      FROM fact_prevalence
  );
"""

with engine.begin() as conn:
    conn.execute(text(create_priority_barriers_view))

print("vw_priority_barriers created successfully.")

vw_priority_barriers created successfully.


### Validate Priority Barriers View

The priority barriers view is queried to check the ranking of access barriers by population group.

In [15]:
priority_barriers = pd.read_sql(
    """
    SELECT *
    FROM vw_priority_barriers
    ORDER BY population, prevalence_rank;
    """,
    engine
)

priority_barriers

,indicator_id,indicator_name,indicator_short_name,barrier_type,population,year,group,total_value,estimated_number,prevalence_rank,estimated_number_rank
0,6,Unmet need for dental health care due to cost,Dental cost barrier,Cost,adults,2024,Total,43.0,1775000.0,1,1
1,4,Unmet need for GP due to wait time,GP wait time barrier,Timeliness,adults,2024,Total,25.5,1107000.0,2,2
2,2,Unmet need for GP due to cost,GP cost barrier,Cost,adults,2024,Total,14.9,646000.0,3,3
3,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,adults,2024,Total,10.5,456000.0,4,4
4,5,Unmet need for GP due to work,GP work constraint,Time constraint,adults,2024,Total,8.8,382000.0,5,5
5,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,adults,2024,Total,3.6,155000.0,6,6
6,3,Unmet need for GP due to transport,GP transport barrier,Transport,adults,2024,Total,3.1,134000.0,7,7
7,4,Unmet need for GP due to wait time,GP wait time barrier,Timeliness,children,2024,Total,19.5,190000.0,1,1
8,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,children,2024,Total,6.3,54000.0,2,2
9,5,Unmet need for GP due to work,GP work constraint,Time constraint,children,2024,Total,3.4,33000.0,3,3


### View 3: Equity Gap

This view summarises subgroup comparison results for the selected health access indicators.

It uses adjusted rate ratios to identify which population groups experience higher access barriers compared with reference groups.

A higher adjusted rate ratio indicates a larger access gap between the comparison group and the reference group.

In [16]:
create_equity_gap_view = """
DROP VIEW IF EXISTS vw_equity_gap;

CREATE VIEW vw_equity_gap AS
SELECT
    di.indicator_id,
    fsc.indicator_name,
    di.indicator_short_name,
    di.barrier_type,
    fsc.population,
    fsc.year,
    fsc.comparison,
    fsc.adjusted_rate_ratio,
    fsc.adjusted_rate_ratio_low_ci,
    fsc.adjusted_rate_ratio_high_ci,
    fsc.adjusted_for,
    RANK() OVER (
        PARTITION BY fsc.population
        ORDER BY fsc.adjusted_rate_ratio DESC
    ) AS equity_gap_rank
FROM fact_subgroup_comparison fsc
LEFT JOIN dim_indicator di
    ON fsc.indicator_name = di.indicator_name
WHERE fsc.year = (
    SELECT MAX(year)
    FROM fact_subgroup_comparison
);
"""

with engine.begin() as conn:
    conn.execute(text(create_equity_gap_view))

print("vw_equity_gap created successfully.")

vw_equity_gap created successfully.


### Validate Equity Gap View

The equity gap view is queried to identify the largest subgroup access gaps.

In [17]:
equity_gap = pd.read_sql(
    """
    SELECT *
    FROM vw_equity_gap
    ORDER BY population, equity_gap_rank;
    """,
    engine
)

equity_gap.head(20)

,indicator_id,indicator_name,indicator_short_name,barrier_type,population,year,comparison,adjusted_rate_ratio,adjusted_rate_ratio_low_ci,adjusted_rate_ratio_high_ci,adjusted_for,equity_gap_rank
0,3,Unmet need for GP due to transport,GP transport barrier,Transport,adults,2024,Disabled adults vs non-disabled adults,6.22,4.40,8.79,"Age, gender",1
1,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,adults,2024,Disabled adults vs non-disabled adults,3.05,2.50,3.73,"Age, gender",2
2,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,adults,2024,Disabled adults vs non-disabled adults,2.82,1.90,4.19,"Age, gender",3
3,3,Unmet need for GP due to transport,GP transport barrier,Transport,adults,2024,Most deprived vs least deprived - men,2.61,0.93,7.31,"Age, ethnic group",4
4,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,adults,2024,Pacific women vs non-Pacific women,2.51,1.58,4.00,Age,5
5,3,Unmet need for GP due to transport,GP transport barrier,Transport,adults,2024,Most deprived vs least deprived,2.38,1.31,4.34,"Age, gender, ethnic group",6
6,3,Unmet need for GP due to transport,GP transport barrier,Transport,adults,2024,Most deprived vs least deprived - women,2.19,1.01,4.76,"Age, ethnic group",7
7,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,adults,2024,Pacific vs non-Pacific,2.15,1.37,3.39,"Age, gender",8
8,2,Unmet need for GP due to cost,GP cost barrier,Cost,adults,2024,Disabled adults vs non-disabled adults,2.06,1.74,2.44,"Age, gender",9
9,2,Unmet need for GP due to cost,GP cost barrier,Cost,adults,2024,Pacific men vs non-Pacific men,2.00,1.33,3.02,Age,10


### View 4: Trend Summary

This view summarises long-term changes in health access barriers.

It compares the earliest available year with the latest available year for each indicator and population group.

Only the total group is used in this view to provide a clear high-level trend summary.

A positive percentage point change suggests the barrier increased over time, while a negative change suggests the barrier decreased.

In [20]:
create_trend_summary_view = """
DROP VIEW IF EXISTS vw_trend_summary;

CREATE VIEW vw_trend_summary AS
WITH total_trend AS (
    SELECT
        ft.indicator_name,
        di.indicator_id,
        di.indicator_short_name,
        di.barrier_type,
        ft.population,
        ft."group",
        ft.year,
        ft.percent_value
    FROM fact_trend ft
    LEFT JOIN dim_indicator di
        ON ft.indicator_name = di.indicator_name
    WHERE ft."group" = 'Total'
      AND ft.percent_value IS NOT NULL
),

year_bounds AS (
    SELECT
        indicator_name,
        population,
        MIN(year) AS start_year,
        MAX(year) AS end_year
    FROM total_trend
    GROUP BY indicator_name, population
),

start_values AS (
    SELECT
        tt.indicator_name,
        tt.population,
        tt.year AS start_year,
        tt.percent_value AS start_value
    FROM total_trend tt
    INNER JOIN year_bounds yb
        ON tt.indicator_name = yb.indicator_name
       AND tt.population = yb.population
       AND tt.year = yb.start_year
),

end_values AS (
    SELECT
        tt.indicator_name,
        tt.population,
        tt.year AS end_year,
        tt.percent_value AS end_value
    FROM total_trend tt
    INNER JOIN year_bounds yb
        ON tt.indicator_name = yb.indicator_name
       AND tt.population = yb.population
       AND tt.year = yb.end_year
)

SELECT
    di.indicator_id,
    sv.indicator_name,
    di.indicator_short_name,
    di.barrier_type,
    sv.population,
    'Total' AS "group",
    sv.start_year,
    ev.end_year,
    ev.end_year - sv.start_year AS years_observed,
    sv.start_value,
    ev.end_value,
    ROUND((ev.end_value - sv.start_value)::numeric, 1) AS percentage_point_change,
    ROUND(
        ((ev.end_value - sv.start_value) / NULLIF((ev.end_year - sv.start_year), 0))::numeric,
        2
    ) AS average_annual_pp_change,
    CASE
        WHEN ev.end_value - sv.start_value > 0 THEN 'Increased'
        WHEN ev.end_value - sv.start_value < 0 THEN 'Decreased'
        ELSE 'No change'
    END AS trend_direction
FROM start_values sv
INNER JOIN end_values ev
    ON sv.indicator_name = ev.indicator_name
   AND sv.population = ev.population
LEFT JOIN dim_indicator di
    ON sv.indicator_name = di.indicator_name;
"""

with engine.begin() as conn:
    conn.execute(text(create_trend_summary_view))

print("vw_trend_summary updated successfully.")

vw_trend_summary updated successfully.


### Validate Trend Summary View

The trend summary view is queried to check long-term changes in access barriers.

In [21]:
trend_summary = pd.read_sql(
    """
    SELECT *
    FROM vw_trend_summary
    ORDER BY population, percentage_point_change DESC;
    """,
    engine
)

trend_summary

,indicator_id,indicator_name,indicator_short_name,barrier_type,population,group,start_year,end_year,years_observed,start_value,end_value,percentage_point_change,average_annual_pp_change,trend_direction
0,4,Unmet need for GP due to wait time,GP wait time barrier,Timeliness,adults,Total,2021,2024,3,11.6,25.5,13.9,4.63,Increased
1,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,adults,Total,2016,2024,8,5.0,10.5,5.5,0.69,Increased
2,6,Unmet need for dental health care due to cost,Dental cost barrier,Cost,adults,Total,2013,2024,11,40.6,43.0,2.4,0.22,Increased
3,5,Unmet need for GP due to work,GP work constraint,Time constraint,adults,Total,2021,2024,3,7.3,8.8,1.5,0.50,Increased
4,2,Unmet need for GP due to cost,GP cost barrier,Cost,adults,Total,2011,2024,13,13.6,14.9,1.3,0.10,Increased
5,3,Unmet need for GP due to transport,GP transport barrier,Transport,adults,Total,2011,2024,13,3.4,3.1,-0.3,-0.02,Decreased
6,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,adults,Total,2011,2024,13,7.3,3.6,-3.7,-0.28,Decreased
7,4,Unmet need for GP due to wait time,GP wait time barrier,Timeliness,children,Total,2021,2024,3,8.0,19.5,11.5,3.83,Increased
8,5,Unmet need for GP due to work,GP work constraint,Time constraint,children,Total,2021,2024,3,1.6,3.4,1.8,0.60,Increased
9,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,children,Total,2016,2024,8,4.7,6.3,1.6,0.20,Increased


### View 5: Priority Matrix

This view combines prevalence, equity gap and trend information to support priority setting.

It brings together:
- latest total-group prevalence and estimated affected population
- the largest subgroup equity gap for each indicator and population
- long-term trend direction and average annual change

This view helps distinguish between broad population issues, targeted equity issues and high-priority issues that are both common and worsening.

In [22]:
create_priority_matrix_view = """
DROP VIEW IF EXISTS vw_priority_matrix;

CREATE VIEW vw_priority_matrix AS
WITH latest_priority AS (
    SELECT
        indicator_id,
        indicator_name,
        indicator_short_name,
        barrier_type,
        population,
        year,
        "group",
        total_value,
        estimated_number,
        prevalence_rank,
        estimated_number_rank
    FROM vw_priority_barriers
),

equity_ranked AS (
    SELECT
        indicator_id,
        indicator_name,
        indicator_short_name,
        barrier_type,
        population,
        comparison,
        adjusted_rate_ratio,
        adjusted_rate_ratio_low_ci,
        adjusted_rate_ratio_high_ci,
        adjusted_for,
        ROW_NUMBER() OVER (
            PARTITION BY indicator_name, population
            ORDER BY adjusted_rate_ratio DESC
        ) AS equity_row_number
    FROM vw_equity_gap
),

top_equity_gap AS (
    SELECT
        indicator_id,
        indicator_name,
        population,
        comparison AS top_equity_comparison,
        adjusted_rate_ratio AS max_adjusted_rate_ratio,
        adjusted_rate_ratio_low_ci AS max_adjusted_rate_ratio_low_ci,
        adjusted_rate_ratio_high_ci AS max_adjusted_rate_ratio_high_ci,
        adjusted_for AS equity_adjusted_for
    FROM equity_ranked
    WHERE equity_row_number = 1
),

trend AS (
    SELECT
        indicator_id,
        indicator_name,
        population,
        start_year,
        end_year,
        years_observed,
        start_value,
        end_value,
        percentage_point_change,
        average_annual_pp_change,
        trend_direction
    FROM vw_trend_summary
)

SELECT
    lp.indicator_id,
    lp.indicator_name,
    lp.indicator_short_name,
    lp.barrier_type,
    lp.population,
    lp.year,
    lp.total_value,
    lp.estimated_number,
    lp.prevalence_rank,
    lp.estimated_number_rank,

    teg.top_equity_comparison,
    teg.max_adjusted_rate_ratio,
    teg.max_adjusted_rate_ratio_low_ci,
    teg.max_adjusted_rate_ratio_high_ci,
    teg.equity_adjusted_for,

    tr.start_year,
    tr.end_year,
    tr.years_observed,
    tr.start_value,
    tr.end_value,
    tr.percentage_point_change,
    tr.average_annual_pp_change,
    tr.trend_direction,

    CASE
        WHEN lp.prevalence_rank <= 2 THEN 'High prevalence'
        WHEN lp.prevalence_rank <= 4 THEN 'Medium prevalence'
        ELSE 'Lower prevalence'
    END AS prevalence_level,

    CASE
        WHEN teg.max_adjusted_rate_ratio >= 2 THEN 'High equity gap'
        WHEN teg.max_adjusted_rate_ratio >= 1.5 THEN 'Medium equity gap'
        WHEN teg.max_adjusted_rate_ratio IS NULL THEN 'No equity comparison'
        ELSE 'Lower equity gap'
    END AS equity_gap_level,

    CASE
        WHEN tr.trend_direction = 'Increased'
             AND tr.average_annual_pp_change >= 1 THEN 'Increasing quickly'
        WHEN tr.trend_direction = 'Increased' THEN 'Increasing'
        WHEN tr.trend_direction = 'Decreased' THEN 'Decreasing'
        ELSE 'Stable'
    END AS trend_level,

    CASE
        WHEN lp.prevalence_rank <= 2
             AND teg.max_adjusted_rate_ratio >= 2
             AND tr.trend_direction = 'Increased'
            THEN 'Highest priority'

        WHEN lp.prevalence_rank <= 2
             AND tr.trend_direction = 'Increased'
            THEN 'Broad population priority'

        WHEN teg.max_adjusted_rate_ratio >= 2
             AND tr.trend_direction = 'Increased'
            THEN 'Targeted equity priority'

        WHEN lp.prevalence_rank <= 4
             AND teg.max_adjusted_rate_ratio >= 1.5
            THEN 'Monitor closely'

        ELSE 'Lower immediate priority'
    END AS priority_category

FROM latest_priority lp
LEFT JOIN top_equity_gap teg
    ON lp.indicator_name = teg.indicator_name
   AND lp.population = teg.population
LEFT JOIN trend tr
    ON lp.indicator_name = tr.indicator_name
   AND lp.population = tr.population;
"""

with engine.begin() as conn:
    conn.execute(text(create_priority_matrix_view))

print("vw_priority_matrix created successfully.")

vw_priority_matrix created successfully.


#### Priority Matrix Logic

The priority matrix uses analyst-defined rules to combine three dimensions:

1. Latest prevalence ranking
2. Largest available subgroup equity gap
3. Trend direction and average annual percentage point change

The priority categories are designed to support exploratory service planning and dashboard reporting. They are not official Ministry of Health classifications.

Interpretation:
- Highest priority: high prevalence, high equity gap and increasing trend
- Broad population priority: high prevalence and increasing trend
- Targeted equity priority: high equity gap and increasing trend
- Monitor closely: medium prevalence and medium equity gap
- Lower immediate priority: lower current priority based on the selected criteria

### Validate Priority Matrix View

The priority matrix is queried to check how access barriers are classified by prevalence, equity gap and trend.

In [23]:
priority_matrix = pd.read_sql(
    """
    SELECT *
    FROM vw_priority_matrix
    ORDER BY population, 
             CASE priority_category
                 WHEN 'Highest priority' THEN 1
                 WHEN 'Broad population priority' THEN 2
                 WHEN 'Targeted equity priority' THEN 3
                 WHEN 'Monitor closely' THEN 4
                 ELSE 5
             END,
             prevalence_rank;
    """,
    engine
)

priority_matrix

,indicator_id,indicator_name,indicator_short_name,barrier_type,population,year,total_value,estimated_number,prevalence_rank,estimated_number_rank,...,years_observed,start_value,end_value,percentage_point_change,average_annual_pp_change,trend_direction,prevalence_level,equity_gap_level,trend_level,priority_category
0,6,Unmet need for dental health care due to cost,Dental cost barrier,Cost,adults,2024,43.0,1775000.0,1,1,...,11,40.6,43.0,2.4,0.22,Increased,High prevalence,Lower equity gap,Increasing,Broad population priority
1,4,Unmet need for GP due to wait time,GP wait time barrier,Timeliness,adults,2024,25.5,1107000.0,2,2,...,3,11.6,25.5,13.9,4.63,Increased,High prevalence,Medium equity gap,Increasing quickly,Broad population priority
2,2,Unmet need for GP due to cost,GP cost barrier,Cost,adults,2024,14.9,646000.0,3,3,...,13,13.6,14.9,1.3,0.10,Increased,Medium prevalence,High equity gap,Increasing,Targeted equity priority
3,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,adults,2024,10.5,456000.0,4,4,...,8,5.0,10.5,5.5,0.69,Increased,Medium prevalence,High equity gap,Increasing,Targeted equity priority
4,5,Unmet need for GP due to work,GP work constraint,Time constraint,adults,2024,8.8,382000.0,5,5,...,3,7.3,8.8,1.5,0.50,Increased,Lower prevalence,Medium equity gap,Increasing,Lower immediate priority
5,1,Unfilled prescription due to cost,Prescription cost barrier,Cost,adults,2024,3.6,155000.0,6,6,...,13,7.3,3.6,-3.7,-0.28,Decreased,Lower prevalence,High equity gap,Decreasing,Lower immediate priority
6,3,Unmet need for GP due to transport,GP transport barrier,Transport,adults,2024,3.1,134000.0,7,7,...,13,3.4,3.1,-0.3,-0.02,Decreased,Lower prevalence,High equity gap,Decreasing,Lower immediate priority
7,7,Unmet need for mental health care and addictio...,Mental health access barrier,Mental health access,children,2024,6.3,54000.0,2,2,...,8,4.7,6.3,1.6,0.20,Increased,High prevalence,High equity gap,Increasing,Highest priority
8,4,Unmet need for GP due to wait time,GP wait time barrier,Timeliness,children,2024,19.5,190000.0,1,1,...,3,8.0,19.5,11.5,3.83,Increased,High prevalence,Lower equity gap,Increasing quickly,Broad population priority
9,5,Unmet need for GP due to work,GP work constraint,Time constraint,children,2024,3.4,33000.0,3,3,...,3,1.6,3.4,1.8,0.60,Increased,Medium prevalence,High equity gap,Increasing,Targeted equity priority


## Validate SQL Views

The created PostgreSQL views are checked to confirm that they exist and return the expected number of records.

In [24]:
view_list_query = """
SELECT table_name
FROM information_schema.views
WHERE table_schema = 'public'
ORDER BY table_name;
"""

pd.read_sql(view_list_query, engine)

,table_name
0,vw_equity_gap
1,vw_latest_prevalence
2,vw_priority_barriers
3,vw_priority_matrix
4,vw_trend_summary


In [25]:
views = [
    "vw_latest_prevalence",
    "vw_priority_barriers",
    "vw_equity_gap",
    "vw_trend_summary",
    "vw_priority_matrix"
]

for view_name in views:
    query = f"SELECT COUNT(*) AS row_count FROM {view_name};"
    result = pd.read_sql(query, engine)
    print(view_name, result.loc[0, "row_count"])

vw_latest_prevalence 14
vw_priority_barriers 14
vw_equity_gap 186
vw_trend_summary 14
vw_priority_matrix 14


## Export SQL Views for Power BI

The validated PostgreSQL views are exported as CSV files for Power BI dashboard development.

Although the reporting tables and views were created in PostgreSQL, exporting the final analytical views as CSV files makes the dashboard workflow easier to transfer across devices.

The exported files will be stored in `data/powerbi`.

In [26]:
powerbi_path = Path("../data/powerbi")
powerbi_path.mkdir(parents=True, exist_ok=True)

print("Power BI export folder created:", powerbi_path)

Power BI export folder created: ../data/powerbi


### Export Reporting Views

Each PostgreSQL view is queried and saved as a CSV file.

These files will be used as the main Power BI data sources.

In [27]:
views_to_export = [
    "vw_latest_prevalence",
    "vw_priority_barriers",
    "vw_equity_gap",
    "vw_trend_summary",
    "vw_priority_matrix"
]

for view_name in views_to_export:
    query = f"SELECT * FROM {view_name};"
    df = pd.read_sql(query, engine)
    
    output_file = powerbi_path / f"{view_name}.csv"
    df.to_csv(output_file, index=False)
    
    print(f"{view_name} exported:", df.shape)

vw_latest_prevalence exported: (14, 13)
vw_priority_barriers exported: (14, 11)
vw_equity_gap exported: (186, 12)
vw_trend_summary exported: (14, 14)
vw_priority_matrix exported: (14, 27)


### Export Trend Fact Table

The full trend fact table is also exported because it contains yearly values needed for line charts in Power BI.

The trend summary view is useful for high-level change analysis, while `fact_trend` supports detailed year-by-year visualisation.

In [28]:
fact_trend_export = pd.read_sql(
    """
    SELECT *
    FROM fact_trend;
    """,
    engine
)

fact_trend_export.to_csv(
    powerbi_path / "fact_trend.csv",
    index=False
)

print("fact_trend exported:", fact_trend_export.shape)

fact_trend exported: (7448, 11)


### Validate Exported Power BI Files

The exported CSV files are checked to confirm that all required Power BI source files were created.

In [29]:
exported_files = sorted(powerbi_path.glob("*.csv"))

for file in exported_files:
    print(file.name)

fact_trend.csv
vw_equity_gap.csv
vw_latest_prevalence.csv
vw_priority_barriers.csv
vw_priority_matrix.csv
vw_trend_summary.csv


In [30]:
for file in exported_files:
    df = pd.read_csv(file)
    print(file.name, df.shape)

fact_trend.csv (7448, 11)
vw_equity_gap.csv (186, 12)
vw_latest_prevalence.csv (14, 13)
vw_priority_barriers.csv (14, 11)
vw_priority_matrix.csv (14, 27)
vw_trend_summary.csv (14, 14)
